# 01 — EDA (T03 e T04)

Dataset: Breast Cancer Wisconsin (`data/data.csv`).  
Alvo clínico: **maligno vs benigno**. Dicionário: [`docs/dicionario_colunas.md`](../docs/dicionario_colunas.md).

Não há scaler, split nem treino aqui (T06–T09). **O médico tem a palavra final.**

## T03 — Carregar a base e mapear características

In [ ]:
from pathlib import Path
import sys

import pandas as pd

ROOT = Path.cwd() if (Path.cwd() / "data" / "data.csv").exists() else Path.cwd().parent
sys.path.insert(0, str(ROOT))

from src.eda import load_frame, split_xy, summarize, save_figures

df = load_frame(ROOT / "data" / "data.csv")
info = summarize(df)
X, y = split_xy(df)

print("shape bruto (linhas, colunas):", info["shape_bruto"])
print("shape de X (sem id, sem diagnosis):", info["shape_X"])
print("id em X?", info["id_em_X"])
print("n features:", len(info["features"]))
print("\ndtypes:")
print(df.dtypes)
print("\ncardinalidade (nunique):")
print(df.nunique().sort_values())
print("\namostra:")
display(df.head(5))
print("\nlista final de features (X):")
print(info["features"])

### Dicionário (resumo)

- `id` — identificador do exame; **fora de X**.
- `diagnosis` — `M` maligno / `B` benigno; é o alvo, não feature.
- 10 famílias × (mean, se, worst) = **30** medidas de núcleo (raio, textura, perímetro, área, smoothness, compactness, concavity, concave points, symmetry, fractal dimension).
- Detalhe clínico de cada família: `docs/dicionario_colunas.md`.

## T04 — Estatísticas descritivas e visualizações

In [ ]:
print("alvo (1=maligno, 0=benigno):")
print(y.value_counts().rename({1: "maligno", 0: "benigno"}))
print(y.value_counts(normalize=True).rename({1: "maligno", 0: "benigno"}).round(4))
print("\ndescribe numérico (X):")
desc = X.describe().T
display(desc)
print("\nmínimos (nenhum deve ser negativo em raio/área/perímetro):")
print(X[["radius_mean", "area_mean", "perimeter_mean"]].min())

In [ ]:
paths = save_figures(df, ROOT / "reports" / "figures")
for p in paths:
    print(p.relative_to(ROOT))

### Figuras geradas

![Contagem do alvo](../reports/figures/01_alvo_contagem.png)

![Box-plots por diagnóstico](../reports/figures/01_boxplots_por_diagnostico.png)

![Histogramas por classe](../reports/figures/01_histogramas_por_classe.png)

![Pairplot do subset](../reports/figures/01_pairplot_subset.png)

### Discussão clínica (não só “o gráfico mostra X”)

**Contagem do alvo.** 357 benignos (62,7%) e 212 malignos (37,3%). A classe que o hospital mais precisa não perder — o maligno — é minoria. Uma regra “chutar sempre benigno” acerta ~63% e é inútil na triagem. Na T12, accuracy sozinha não manda; recall do maligno é candidata a métrica-guia.

**Box-plots (worst concave points, perímetro/raio/área worst, concave points e radius mean).** Núcleos malignos tendem a ser **maiores** e com **mais indentação** do contorno. Isso combina com a citologia: atipia de tamanho e irregularidade de borda. Ainda assim as caixas se sobrepõem: um limiar único em `radius_mean` ou em `concave points_worst` erra casos de fronteira. Daí modelo probabilístico + médico no final, não corte automático.

**Histogramas.** A massa do maligno em `concave points_worst` e `radius_mean` está à direita da do benigno, mas a interseção é larga. Overlap = falso positivo e falso negativo inevitáveis se alguém “olhar só o gráfico”. SHAP (T14) explica o caso; não prova biologia.

**Pairplot (subset).** Mesmo em 4 eixos os grupos não se separam como duas ilhas. Correlação entre raio e área é esperada (T07 trata pares colineares). Nada aqui autoriza dropar feature ainda, nem treinar “só para ver” (queimaria o teste mentalmente).

**Anomalias para a T05.** Mínimos de raio/área/perímetro são positivos neste CSV; não há zero sentinela tipo Pima. Faltantes e duplicatas se checam na limpeza, não aqui.

**O médico tem a palavra final.** Estas figuras priorizam o que o modelo deve enxergar; não fecham diagnóstico.